# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Which content and search signals — measured from
first-half monthly data — are associated with a page losing search
visibility in the second half of the same month? And can those signals,
combined into a model, identify at-risk pages early enough to be
actionable?

**Decision this supports:** a content strategist prioritizes which pages
to review for refresh, title/CTA adjustment, or deeper investigation,
rather than reviewing pages in publish-date order or at random.

**Unit of analysis:** one content page (`content_hash_id`) per month,
measured over a 31-day window (first-half features → second-half label).

**Output:** a ranked priority queue of pages, each labeled with a reason
code and a recommended action.

**Cost of a wrong call:** over-flagging wastes reviewer time on stable
pages; under-flagging lets genuinely declining pages keep losing traffic
until the drop is too large to recover quickly. Both errors have real
operational cost — the triage tool must improve over the unaided
hand-written rule to be worth using.

**Why data / ML helps here at all:** with ~60,000+ content items and
weak individual signals (no single metric predicts decline cleanly), a
model that combines several noisy signals outperforms any fixed
single-metric rule — as demonstrated by the w04 baseline experiment.

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# ── use the _sample table for fast iteration; swap to full table for
# the final run when the query is locked
SAMPLE_TABLE = f"{rel}/fact_content_daily_performance_sample/**/*.parquet"
FULL_TABLE   = f"{rel}/fact_content_daily_performance/**/*.parquet"

# WORKING MONTH — mid-panel, not the sealed final month (June 2026)
MONTH = "2026-03"

print("Setup complete. Working month:", MONTH)
print("Warehouse tables available:")
tables = ["dim_clients", "dim_content", "fact_content_daily_performance",
          "fact_content_query_90d"]
for t in tables:
    print(" •", t)

Setup complete. Working month: 2026-03
Warehouse tables available:
 • dim_clients
 • dim_content
 • fact_content_daily_performance
 • fact_content_query_90d


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank ML Internship Warehouse (build v20260703),
Hugging Face (`FlyRank/internship-warehouse`), gated access.

**Tables used:**
- `fact_content_daily_performance` (partitioned by month) — daily
  GSC impressions, clicks, avg_position, availability flags.
- `dim_content` — static content metadata (word_count,
  content_created_date, content_type, main_intent).

**Date window:** March 2026 (`month = 2026-03`), mid-panel month.
June 2026 is treated as a sealed test month and never used for
feature or label development.

**What was excluded, and why:**
- All `ga4_*` columns (pageviews, sessions, engagement, AI traffic
  breakdown) — this lane is about search signal analysis specifically;
  on-site engagement belongs to a different lane.
- Rows where `gsc_data_available IS NOT TRUE` — these contain
  zero-filled GSC columns that represent missing data, not real
  measurements (confirmed: 36.7% of March 2026 rows are excluded
  by this filter, per the Week 3 data contract).
- `trend_direction` and `trend_pct` — label-derived fields; never
  features (the label `is_declining` is derived from these, so
  including them as features is direct leakage).
- `client_hash_id` — used only for grouped train/test splitting,
  never as a model feature.
- Rows with `content_age_days < 0` — timestamp artifact (5.1% of
  joined rows in March 2026, Week 3 finding); excluded by filter.
- Content items with fewer than 50 first-half impressions —
  too sparse for a reliable position/CTR observation.

**No client names, domains, raw queries, or identifying details appear
anywhere in this notebook or its outputs.**

In [2]:
# Load and verify the feature frame — same query as w03/w04/w05/w06
df = con.sql(f"""
WITH daily AS (
    SELECT *
    FROM read_parquet('{FULL_TABLE}', hive_partitioning=1)
    WHERE month = '{MONTH}' AND gsc_data_available IS TRUE
),
first_half AS (
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_avg_position)  AS avg_position,
           SUM(gsc_impressions)   AS impressions,
           SUM(gsc_clicks)        AS clicks
    FROM daily
    WHERE report_date <= DATE '{MONTH}-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 50
),
second_half AS (
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_avg_position)  AS sh_pos
    FROM daily
    WHERE report_date > DATE '{MONTH}-15'
    GROUP BY 1, 2
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.avg_position,
    f.impressions,
    f.clicks,
    c.word_count,
    DATE_DIFF('day', c.content_created_date, DATE '{MONTH}-01') AS content_age_days,
    s.sh_pos,
    CASE WHEN s.sh_pos > f.avg_position THEN 1 ELSE 0 END AS is_declining
FROM first_half f
JOIN second_half s USING (client_hash_id, content_hash_id)
JOIN read_parquet('{rel}/dim_content.parquet') c
     USING (client_hash_id, content_hash_id)
WHERE f.avg_position IS NOT NULL AND s.sh_pos IS NOT NULL
""").df()

df["actual_ctr"] = df["clicks"] / df["impressions"]
df = df.dropna(subset=["word_count", "content_age_days"])
df = df[df["content_age_days"] >= 0]

print("Rows after filtering:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())
print("Label balance — declining:", df["is_declining"].mean().round(3))
print("GSC availability (March 2026):", "36.7% of raw rows (Week 3)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after filtering: 60645
Unique clients: 37
Label balance — declining: 0.531
GSC availability (March 2026): 36.7% of raw rows (Week 3)


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label definition:** `is_declining = 1` if a page's average position in
the second half of March 2026 is numerically worse (higher) than in the
first half; else 0. This is a within-month proxy for ranking decay —
directional, not causal.

**Features (all knowable at the first-half decision point):**
1. `avg_position` — average GSC position, days 1-15
2. `impressions` — total GSC impressions, days 1-15
3. `actual_ctr` — clicks ÷ impressions, days 1-15
4. `word_count` — from `dim_content`, static snapshot
5. `content_age_days` — days since `content_created_date` at
   month start; ages < 0 excluded

**Baseline (Week 4):** a hand-written rule scoring pages by
`(benchmark_ctr − actual_ctr) × impressions`, where `benchmark_ctr`
is the mean CTR for that page's position bucket (1-3, 4-10, 11-20, 21+).
This is the simplest defensible rule: if your CTR is below what pages
ranked similarly typically earn, you are flagged.

**Validation design:** grouped by `client_hash_id` — all rows for one
client go entirely to train or entirely to test (70/30 client split).
This prevents the ~5.7% precision overstatement observed when using a
naive random row split (Week 6 finding). Fixed seed (42) for
reproducibility. No time-aware split was used because all rows come
from the same month; within-month grouped split is the appropriate
honest design here.

**Models evaluated:** Logistic Regression and Random Forest, both
trained on the five features above.

**Success metric:** Precision@top-20% — of the pages the model ranks
highest by predicted decline probability, what fraction are actually
declining? This was chosen over accuracy because: (a) accuracy looks
artificially good given ~57% base rate, and (b) the content team can
only review a shortlist, not all 60,000+ pages.

**Leakage checks:**
- `trend_direction` and `trend_pct` never appear as features.
- `sh_pos` (second-half position, the label's source) is excluded
  from features and confirmed absent (Week 3/6 audit).
- `client_hash_id` is grouping-only, never a feature.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

features = ["avg_position", "impressions", "actual_ctr",
            "word_count", "content_age_days"]

# Leakage check
leaked = [f for f in features if f in ["sh_pos","is_declining",
                                        "trend_direction","trend_pct"]]
print("Label-derived columns in features:", leaked if leaked else "None ✓")
print("client_hash_id in features:", "client_hash_id" in features)

# Grouped split
rng = np.random.RandomState(42)
clients = np.sort(df["client_hash_id"].unique())
rng.shuffle(clients)
split_pt = int(len(clients) * 0.7)
tr_clients, te_clients = clients[:split_pt], clients[split_pt:]
train = df[df["client_hash_id"].isin(tr_clients)]
test  = df[df["client_hash_id"].isin(te_clients)]
print(f"\nTrain: {len(train)} rows, {len(tr_clients)} clients")
print(f"Test:  {len(test)} rows, {len(te_clients)} clients")
print("Client overlap:", len(set(tr_clients) & set(te_clients)), "✓")

# Position bucket and baseline — compute on full df BEFORE splitting
df["position_bucket"] = pd.cut(
    df["avg_position"], bins=[0,3,10,20,10000],
    labels=["1-3","4-10","11-20","21+"])
bm = df.groupby("position_bucket", observed=True)["actual_ctr"].mean()

# NOW do the split (so test inherits position_bucket)
rng = np.random.RandomState(42)
clients = np.sort(df["client_hash_id"].unique())
rng.shuffle(clients)
split_pt = int(len(clients) * 0.7)
tr_clients, te_clients = clients[:split_pt], clients[split_pt:]
train = df[df["client_hash_id"].isin(tr_clients)]
test  = df[df["client_hash_id"].isin(te_clients)]
print(f"\nTrain: {len(train)} rows, {len(tr_clients)} clients")
print(f"Test:  {len(test)} rows, {len(te_clients)} clients")
print("Client overlap:", len(set(tr_clients) & set(te_clients)), "✓")

# Baseline scores on test set
test_bm = test["position_bucket"].map(bm).astype(float)
baseline_scores = (test_bm - test["actual_ctr"]) * test["impressions"]

def p_at_k(y, scores, pct=0.20):
    k = max(1, int(len(y) * pct))
    idx = np.argsort(scores.values)[::-1][:k]
    return precision_score(np.array(y)[idx], np.ones(k))

base_rate = test["is_declining"].mean()
baseline_p = p_at_k(test["is_declining"], baseline_scores)
print(f"\nBase rate: {base_rate:.3f}")
print(f"Baseline Precision@20%: {baseline_p:.3f}")

Label-derived columns in features: None ✓
client_hash_id in features: False

Train: 46140 rows, 25 clients
Test:  14505 rows, 12 clients
Client overlap: 0 ✓

Train: 46140 rows, 25 clients
Test:  14505 rows, 12 clients
Client overlap: 0 ✓

Base rate: 0.579
Baseline Precision@20%: 0.587


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
X_tr, y_tr = train[features], train["is_declining"]
X_te, y_te = test[features],  test["is_declining"]

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr, y_tr)
rf     = RandomForestClassifier(n_estimators=200, max_depth=6,
                                 random_state=42).fit(X_tr, y_tr)

lr_scores = logreg.predict_proba(X_te)[:, 1]
rf_scores = rf.predict_proba(X_te)[:, 1]

results = pd.DataFrame({
    "Method":            ["Base rate", "Baseline rule (w04)",
                          "Logistic Regression", "Random Forest"],
    "Precision@20%":     [round(base_rate, 3),
                          round(baseline_p, 3),
                          round(p_at_k(y_te, pd.Series(lr_scores)), 3),
                          round(p_at_k(y_te, pd.Series(rf_scores)), 3)],
})
print(results.to_string(index=False))

# Coefficient table (LR)
print("\nLogistic Regression coefficients:")
for f, c in sorted(zip(features, logreg.coef_[0]),
                   key=lambda x: abs(x[1]), reverse=True):
    print(f"  {f}: {c:+.4f}")

# Save for paper
os.makedirs("work/outputs", exist_ok=True)
results.to_csv("work/outputs/capstone_results.csv", index=False)
print("\nSaved: work/outputs/capstone_results.csv")

             Method  Precision@20%
          Base rate          0.579
Baseline rule (w04)          0.587
Logistic Regression          0.680
      Random Forest          0.714

Logistic Regression coefficients:
  avg_position: -0.0580
  actual_ctr: +0.0019
  content_age_days: -0.0012
  word_count: -0.0001
  impressions: +0.0000

Saved: work/outputs/capstone_results.csv


## 5. Limitations

*What this work cannot claim.*

In [5]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(logreg, X_te, y_te,
                              n_repeats=15, random_state=42)
print("Permutation importance (Logistic Regression):")
for f, imp in sorted(zip(features, perm.importances_mean),
                     key=lambda x: -x[1]):
    print(f"  {f}: {imp:+.4f}")

# Client-concentration check
test_copy = test.copy()
test_copy["decline_risk"] = lr_scores
top20_pct = test_copy.sort_values("decline_risk", ascending=False).head(
    int(len(test_copy) * 0.20))
conc = top20_pct["client_hash_id"].value_counts(normalize=True).head(3)
print("\nTop-3 client concentration in top-20% flag list:")
print(conc)
print("(High concentration → possible client-level issue, not content issue)")

Permutation importance (Logistic Regression):
  avg_position: +0.0679
  word_count: +0.0012
  actual_ctr: +0.0000
  impressions: -0.0003
  content_age_days: -0.0075

Top-3 client concentration in top-20% flag list:
client_hash_id
client_73cda7b4e4f265ea    0.661841
client_e5c2aa26a8598242    0.251982
client_20259bd6705d81d4    0.068597
Name: proportion, dtype: float64
(High concentration → possible client-level issue, not content issue)


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The playbook is a ranked queue combining two signals: the validated
CTR rule (Week 4, CONFIRMED signal) and the model's decline-risk score
(Week 5/6, Precision@20% = see results table). Each page gets one
reason code and one action.

**Archetypes and actions:**

| Archetype | Signal pattern | Action |
|---|---|---|
| At-Risk Leader | Good position, high decline risk | `review_content_refresh` — protect first |
| CTR Underperformer | CTR below position benchmark | `review_ctr_fix` — title/snippet rewrite |
| Monitor | Moderate risk, not yet actionable | `monitor_next_cycle` |
| Stable | Low decline risk, CTR on par | `no_action` |

**How a content strategist uses this tomorrow:**
1. Pull the `capstone_queue.csv` from this notebook's exports.
2. Filter to `action != 'no_action'`.
3. Start with At-Risk Leaders (well-ranked pages at risk — highest
   opportunity cost if missed).
4. For each: confirm the signal is real (not a tracking artifact),
   then investigate whether a content update, title refresh, or
   structural change is warranted.
5. Report back any pattern the model missed — that feedback informs
   the next retrain.

**Confidence and limits:** Precision@20% on the honest split means
roughly 1 in 3 flagged pages in the top-20% shortlist is not actually
declining. This is a triage tool, not a certainty — human review is
mandatory before acting on any individual recommendation.

In [6]:
df_queue = df.copy()
lr_all = logreg.predict_proba(df_queue[features])[:, 1]
df_queue["decline_risk_score"] = lr_all
bm_all = df_queue["position_bucket"].map(bm).astype(float)
df_queue["ctr_gap"] = bm_all - df_queue["actual_ctr"]

def assign(row):
    if row["decline_risk_score"] > 0.5 and row["avg_position"] <= 10:
        return "At-Risk Leader", "review_content_refresh"
    elif row["ctr_gap"] > 0 and row["decline_risk_score"] > 0.4:
        return "CTR Underperformer", "review_ctr_fix"
    elif row["decline_risk_score"] > 0.5:
        return "Monitor", "monitor_next_cycle"
    else:
        return "Stable", "no_action"

df_queue[["archetype","action"]] = df_queue.apply(
    lambda r: pd.Series(assign(r)), axis=1)

queue = (df_queue[df_queue["action"] != "no_action"]
         .sort_values("decline_risk_score", ascending=False)
         [[
           "client_hash_id","content_hash_id","archetype",
           "position_bucket","avg_position","decline_risk_score",
           "ctr_gap","action"
         ]])

print("Action queue size:", len(queue))
print(queue["archetype"].value_counts())
print("\nTop 5:")
print(queue.head(5).to_string(index=False))

queue.to_csv("work/outputs/capstone_queue.csv", index=False)
print("\nSaved: work/outputs/capstone_queue.csv")

Action queue size: 46107
archetype
At-Risk Leader        34667
CTR Underperformer     8688
Monitor                2752
Name: count, dtype: int64

Top 5:
         client_hash_id          content_hash_id      archetype position_bucket  avg_position  decline_risk_score   ctr_gap                 action
client_2094c6eb080311d5 content_c886c414b098c87f At-Risk Leader             1-3      1.436881            0.752879 -0.049309 review_content_refresh
client_fef1a8f436438636 content_0a02244dbd14c74f At-Risk Leader             1-3      0.629415            0.751070  0.000767 review_content_refresh
client_fef1a8f436438636 content_eac0a973f63c8b0d At-Risk Leader             1-3      0.163043            0.749452  0.004089 review_content_refresh
client_fef1a8f436438636 content_695fb13c42a38a74 At-Risk Leader             1-3      1.369887            0.748658 -0.003515 review_content_refresh
client_fef1a8f436438636 content_a8fae87fcb797566 At-Risk Leader             1-3      0.507278            0.74639

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import json

os.makedirs("work/figures", exist_ok=True)

# ── Figure 1: model vs baseline bar chart
fig, ax = plt.subplots(figsize=(7, 3.5))
r = results.copy()
bars = ax.bar(r["Method"], r["Precision@20%"],
              color=["#9CA3AF","#6B7280","#2F6E6A","#1A4F4C"])
ax.axhline(r.loc[r["Method"]=="Base rate","Precision@20%"].values[0],
           color="#D1D5DB", linestyle="--", linewidth=1)
ax.set_ylim(0, 1)
ax.set_ylabel("Precision @ top 20%")
ax.set_title("Model vs. Baseline — March 2026 (grouped client split)")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("work/figures/capstone_model_vs_baseline.png", dpi=150)
plt.close()
print("Saved: work/figures/capstone_model_vs_baseline.png")

# ── Figure 2: archetype distribution
fig, ax = plt.subplots(figsize=(6, 3))
df_queue["archetype"].value_counts().plot(kind="bar", ax=ax,
                                           color="#2F6E6A")
ax.set_ylabel("Content items")
ax.set_title("Archetype distribution — action queue")
plt.tight_layout()
plt.savefig("work/figures/capstone_archetypes.png", dpi=150)
plt.close()
print("Saved: work/figures/capstone_archetypes.png")

# ── Metrics JSON (committed receipt)
metrics = {
    "month": MONTH,
    "n_rows": int(len(df)),
    "n_clients": int(df["client_hash_id"].nunique()),
    "base_rate": round(float(base_rate), 4),
    "baseline_precision_at_20pct": round(float(baseline_p), 4),
    "logreg_precision_at_20pct": round(float(p_at_k(y_te, pd.Series(lr_scores))), 4),
    "rf_precision_at_20pct": round(float(p_at_k(y_te, pd.Series(rf_scores))), 4),
    "features": features,
    "split": "grouped_by_client_70_30",
    "random_seed": 42,
}
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved: work/outputs/capstone_metrics.json")
print("\nAll artifacts generated.")

Saved: work/figures/capstone_model_vs_baseline.png
Saved: work/figures/capstone_archetypes.png
Saved: work/outputs/capstone_metrics.json

All artifacts generated.


## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [ ✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.